# Sendo a base apresentada no arquivo abaixo:
- https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce?select=olist_customers_dataset.csv
    - Já disponível no banco de dados `vendas_db.db` através do link:
        - https://drive.google.com/file/d/1eONzrbEu5BoijDRj56gjL_2Ik5qxtR6U/view?usp=sharing
<br><br>
- Sua tarefa é ajudar a área de negócios a **montar uma apresentação para a diretoria** para provar a **necessidade de investistimento em uma área de melhoria da experência do cliente ao ter um atraso na entrega**
<br><br>
- Algumas considerações são importantes
    - O **time de logística não considera que o atraso na entrega é um problema relevante** e falou que, em média, as entregas estão sendo feitas 10 dias antes do prazo combinado
    - Não é desejado a previsão de uma entrega atrasada, apenas a **exposição que esse é um problema que pode impactar os clientes**
    - Não queremos uma abordagem de: "nenhuma entrega pode atrasar". Vamos ser mais tranquilos e seguir na linha de: **"uma entrega pode atrasar. Como eu posso melhorar a experiência do cliente caso isso aconteça?"**

In [4]:
import sqlite3 
import pandas as pd

con = sqlite3.connect('../data/vendas_db.db')

cur = con.cursor()

In [5]:
def executa_consulta(consulta):
    resultado = cur.execute(consulta).fetchall()
    resultado = pd.DataFrame(resultado)
    colunas = [i[0] for i in cur.description]
    if resultado.shape[1] > 0:
        resultado.columns = colunas
    print(resultado.shape)
    display(resultado.head(3))
    return resultado

In [6]:
executa_consulta('SELECT * FROM orders')

(99441, 9)


,index,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


,index,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...,...
99436,99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [12]:
executa_consulta('SELECT COUNT (*) AS total_pedidos_entregues,\
                 ROUND(AVG(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date)), 2) AS media_dias_vs_prazo\
                 FROM orders\
                 WHERE order_status = "delivered"\
                 AND order_delivered_customer_date IS NOT NULL\
                 AND order_estimated_delivery_date IS NOT NULL')

(1, 2)


,total_pedidos_entregues,media_dias_vs_prazo
0,96470,-11.18


,total_pedidos_entregues,media_dias_vs_prazo
0,96470,-11.18


### PASSO 1 

“A logística está correta quando olha a média: os pedidos entregues chegaram, em média, 11 dias antes do prazo. Porém, experiência do cliente não acontece na média. Ela acontece pedido a pedido. Por isso, o próximo passo é separar entregas no prazo e entregas atrasadas.”



## Se a média é boa, ainda assim existem clientes impactados por atraso?


In [13]:
executa_consulta('SELECT \
                 CASE \
                     WHEN order_delivered_customer_date > order_estimated_delivery_date \
                         THEN "Atrasado" \
                     ELSE "No prazo ou adiantado" \
                 END AS status_entrega, \
                 COUNT(*) AS qtd_pedidos, \
                 ROUND(100.0 * COUNT(*) / ( \
                     SELECT COUNT(*) \
                     FROM orders \
                     WHERE order_status = "delivered" \
                       AND order_delivered_customer_date IS NOT NULL \
                       AND order_estimated_delivery_date IS NOT NULL \
                 ), 2) AS pct_pedidos \
                 FROM orders \
                 WHERE order_status = "delivered" \
                   AND order_delivered_customer_date IS NOT NULL \
                   AND order_estimated_delivery_date IS NOT NULL \
                 GROUP BY status_entrega')

(2, 3)


,status_entrega,qtd_pedidos,pct_pedidos
0,Atrasado,7826,8.11
1,No prazo ou adiantado,88644,91.89


,status_entrega,qtd_pedidos,pct_pedidos
0,Atrasado,7826,8.11
1,No prazo ou adiantado,88644,91.89


“A operação realmente funciona bem para a maioria dos clientes: quase 92% dos pedidos chegam no prazo ou antes. Mas ainda temos 7.826 clientes impactados por atraso. A proposta aqui não é dizer que a logística falha como um todo, e sim que existe um grupo relevante de clientes que precisa de uma experiência melhor quando o atraso acontece.”

## Agora que sabemos que o atraso existe, precisamos entender se ele afeta a satisfação do cliente.
